# <font color="#418FDE" size="6.5" uppercase>**Kernel Approximation**</font>

>Last update: 20260712.
    
By the end of this Lecture, you will be able to:
- Apply kernel approximation methods to create explicit feature maps for nonlinear models. 
- Use random projection methods to reduce dimensionality while approximately preserving distances. 
- Evaluate approximation quality by comparing runtime, transformed dimensions, and downstream scores. 


## **1. Kernel Feature Maps**

### **1.1. Nyström Approximation**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_08/Lecture_B/image_01_01.jpg?v=1783839056" width="250">



>* Landmarks create cheaper explicit kernel features.
>* Linear models still capture nonlinear patterns.

>* Nyström compares samples to representative landmarks.
>* Good landmark coverage preserves nonlinear structure.

>* Linear models gain kernel-like speed and flexibility.
>* Landmark choices balance accuracy and efficiency.



In [ ]:
#@title Python Code - Nyström Approximation

# This script demonstrates Nyström style kernel features.
# It compares exact and approximate similarities.
# Small data keeps the example beginner friendly.

import numpy as np
import matplotlib.pyplot as plt

# Set a deterministic random seed.
rng = np.random.default_rng(7)

# Create two curved toy classes.
angles_a = rng.uniform(0.0, np.pi, 60)
angles_b = rng.uniform(0.0, np.pi, 60)

# Build moon like coordinates.
x_a = np.c_[np.cos(angles_a), np.sin(angles_a)]
x_b = np.c_[1.0 - np.cos(angles_b), 0.5 - np.sin(angles_b)]

# Add small noise for realism.
noise_a = rng.normal(0.0, 0.08, x_a.shape)
noise_b = rng.normal(0.0, 0.08, x_b.shape)

# Combine points and labels.
X = np.vstack([x_a + noise_a, x_b + noise_b])
y = np.r_[np.zeros(60, dtype=int), np.ones(60, dtype=int)]

# Define an RBF kernel function.
def rbf_kernel(A, B, gamma):
    diff = A[:, None, :] - B[None, :, :]
    sqdist = np.sum(diff * diff, axis=2)
    return np.exp(-gamma * sqdist)

# Choose kernel settings and landmarks.
gamma = 2.0
m = 20
indices = rng.choice(X.shape[0], size=m, replace=False)

# Build Nyström feature map pieces.
landmarks = X[indices]
C = rbf_kernel(X, landmarks, gamma)
W = rbf_kernel(landmarks, landmarks, gamma)

# Stabilize the landmark kernel matrix.
W = W + 1e-8 * np.eye(m)
eigvals, eigvecs = np.linalg.eigh(W)

# Form explicit approximate features.
inv_sqrt = np.diag(1.0 / np.sqrt(np.maximum(eigvals, 1e-10)))
Z = C @ eigvecs @ inv_sqrt

# Compare exact and approximate kernels.
K_exact = rbf_kernel(X, X, gamma)
K_approx = Z @ Z.T
rel_error = np.linalg.norm(K_exact - K_approx) / np.linalg.norm(K_exact)

# Summarize the transformed representation.
print(f"Original input dimensions: {X.shape[1]}")
print(f"Nyström feature dimensions: {Z.shape[1]}")
print(f"Landmarks used: {m}")
print(f"Relative kernel error: {rel_error:.3f}")

# Plot original data and feature map.
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", s=25)

# Mark selected landmarks clearly.
axes[0].scatter(landmarks[:, 0], landmarks[:, 1], facecolors="none", edgecolors="black", s=90)
axes[0].set_title("Original data with landmarks")
axes[0].set_xlabel("x1")

# Show first two explicit features.
axes[1].scatter(Z[:, 0], Z[:, 1], c=y, cmap="coolwarm", s=25)
axes[1].set_title("First two Nyström features")
axes[1].set_xlabel("feature 1")

# Finish the second panel.
axes[1].set_ylabel("feature 2")
axes[0].set_ylabel("x2")
plt.tight_layout()
plt.show()



### **1.2. Random Fourier Features**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_08/Lecture_B/image_01_02.jpg?v=1783839054" width="250">



>* Approximate kernels with explicit random features.
>* Enable faster linear models with nonlinear patterns.

>* Random waves approximate kernel-based similarity patterns.
>* Enables scalable linear learning on large datasets.

>* More features improve accuracy but cost more.
>* Useful when kernel choice and size fit.



In [ ]:
#@title Python Code - Random Fourier Features

# Random features approximate nonlinear kernels explicitly.
# This example builds Fourier feature maps.
# Then linear regression models curved patterns.

import numpy as np
import matplotlib.pyplot as plt

# Set a deterministic random seed.
rng = np.random.default_rng(7)

# Create a tiny nonlinear dataset.
x = np.linspace(-3.0, 3.0, 80).reshape(-1, 1)
y = np.sin(2.0 * x[:, 0]) + 0.15 * rng.normal(size=80)

# Split data into train and test.
train_mask = np.arange(80) < 60
x_train, x_test = x[train_mask], x[~train_mask]
y_train, y_test = y[train_mask], y[~train_mask]

# Build random Fourier feature maps.
def rff_map(x_data, n_features, gamma, seed):

    # Validate the requested feature count.
    if n_features < 2:
        raise ValueError("n_features must be at least 2")

    # Sample random directions and phases.
    local_rng = np.random.default_rng(seed)
    weights = local_rng.normal(
        loc=0.0, scale=np.sqrt(2.0 * gamma),
        size=(x_data.shape[1], n_features)
    )
    phases = local_rng.uniform(0.0, 2.0 * np.pi, size=n_features)

    # Return the cosine feature matrix.
    projection = x_data @ weights + phases
    return np.sqrt(2.0 / n_features) * np.cos(projection)

# Fit simple linear regression safely.
def fit_linear(features, targets):

    # Add an intercept column.
    design = np.column_stack([np.ones(features.shape[0]), features])

    # Solve with least squares.
    coef, _, _, _ = np.linalg.lstsq(design, targets, rcond=None)
    return coef

# Predict from fitted coefficients.
def predict_linear(features, coef):

    # Add an intercept column.
    design = np.column_stack([np.ones(features.shape[0]), features])
    return design @ coef

# Measure mean squared error.
def mse(y_true, y_pred):

    # Compute a compact error summary.
    return float(np.mean((y_true - y_pred) ** 2))

# Fit a plain linear baseline.
base_coef = fit_linear(x_train, y_train)
base_pred = predict_linear(x_test, base_coef)
base_error = mse(y_test, base_pred)

# Fit regression after Fourier mapping.
gamma = 1.0
n_features = 200
z_train = rff_map(x_train, n_features, gamma, seed=11)
z_test = rff_map(x_test, n_features, gamma, seed=11)
rff_coef = fit_linear(z_train, y_train)
rff_pred = predict_linear(z_test, rff_coef)
rff_error = mse(y_test, rff_pred)

# Build smooth predictions for plotting.
x_grid = np.linspace(-3.0, 3.0, 300).reshape(-1, 1)
base_grid = predict_linear(x_grid, base_coef)
z_grid = rff_map(x_grid, n_features, gamma, seed=11)
rff_grid = predict_linear(z_grid, rff_coef)

# Print a short comparison summary.
print(f"Original shape: {x_train.shape}")
print(f"Mapped shape: {z_train.shape}")
print(f"Linear test MSE: {base_error:.3f}")
print(f"RFF test MSE: {rff_error:.3f}")
print("RFF adds explicit nonlinear features.")

# Plot data and both fitted curves.
plt.figure(figsize=(8, 4))
plt.scatter(x_train[:, 0], y_train, s=18, label="train points")
plt.scatter(x_test[:, 0], y_test, s=18, label="test points")
plt.plot(x_grid[:, 0], base_grid, label="linear baseline")
plt.plot(x_grid[:, 0], rff_grid, label="RFF linear model")

# Finish the single teaching plot.
plt.title("Random Fourier Features for Nonlinear Regression")
plt.xlabel("Input x")
plt.ylabel("Target y")
plt.legend()
plt.tight_layout()
plt.show()



### **1.3. Comparing Feature Maps**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_08/Lecture_B/image_01_03.jpg?v=1783839058" width="250">



>* Feature maps enable linear learning efficiently.
>* Methods differ in construction and performance.

>* Nyström uses sampled data structure directly.
>* RFF scales with uniform randomized features.

>* Balance feature accuracy against training efficiency.
>* Choose maps that fit real-world demands.



## **2. Random Projections**

### **2.1. Random Projection Basics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_08/Lecture_B/image_02_01.jpg?v=1783839048" width="250">



>* Random mixes compress many features into fewer.
>* Usually preserves point relationships for fast learning.

>* Random projection compresses data while keeping similarities.
>* Fewer synthetic features improve efficiency reliably.

>* Fast, simple compression for high-dimensional data.
>* Preserves useful distances for downstream tasks.



In [ ]:
#@title Python Code - Random Projection Basics

# Random projections compress data using random directions.
# Distances often stay similar after projection.
# This example shows that basic idea.

import numpy as np
import matplotlib.pyplot as plt

# Set a deterministic random seed.
rng = np.random.default_rng(7)

# Create simple high dimensional sample data.
X = rng.normal(size=(60, 40))

# Validate the data shape first.
assert X.shape == (60, 40)

# Build a Gaussian random projection matrix.
R = rng.normal(size=(40, 2)) / np.sqrt(2)

# Project data into two dimensions.
Y = X @ R

# Check projected shape for safety.
assert Y.shape == (60, 2)

# Pick two sample points to compare.
i, j = 3, 25

# Compute original Euclidean distance.
d_original = np.linalg.norm(X[i] - X[j])

# Compute projected Euclidean distance.
d_projected = np.linalg.norm(Y[i] - Y[j])

# Report the key comparison briefly.
print(f"Original distance: {d_original:.2f}")
print(f"Projected distance: {d_projected:.2f}")
print(f"Original shape: {X.shape}")
print(f"Projected shape: {Y.shape}")

# Plot the projected points clearly.
plt.figure(figsize=(6, 4))

# Show all projected observations.
plt.scatter(Y[:, 0], Y[:, 1], alpha=0.7)

# Highlight the compared two points.
plt.scatter(Y[[i, j], 0], Y[[i, j], 1], s=90)

# Add a short informative title.
plt.title("Random projection into two dimensions")

# Label the new synthetic axes.
plt.xlabel("Projected feature 1")
plt.ylabel("Projected feature 2")

# Keep the layout neat.
plt.tight_layout()
plt.show()



### **2.2. Distance Preserving Projections**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_08/Lecture_B/image_02_02.jpg?v=1783839050" width="250">



>* Compresses data while keeping distances roughly meaningful.
>* Supports faster analysis for distance based methods.

>* Random directions can preserve pairwise distances.
>* Useful for compressing very high dimensional data.

>* Too few dimensions distort important distances.
>* Balance efficiency with trustworthy downstream structure.



In [ ]:
#@title Python Code - Distance Preserving Projections

# Random projections can preserve distances surprisingly well.
# This script compares distances before and after projection.
# A simple plot shows the preservation pattern.

import numpy as np
import matplotlib.pyplot as plt

# Set a deterministic random seed.
rng = np.random.default_rng(7)

# Create high dimensional sample points.
points = rng.normal(size=(60, 120))

# Validate the generated data shape.
assert points.shape == (60, 120)

# Build a random Gaussian projection matrix.
target_dim = 12
projection = rng.normal(size=(120, target_dim)) / np.sqrt(target_dim)

# Project points into lower dimensions.
projected = points @ projection

# Validate projected data shape.
assert projected.shape == (60, target_dim)

# Compute pairwise distances without extra libraries.
diff_original = points[:, None, :] - points[None, :, :]
diff_projected = projected[:, None, :] - projected[None, :, :]

# Convert differences into Euclidean distances.
dist_original = np.sqrt((diff_original ** 2).sum(axis=2))
dist_projected = np.sqrt((diff_projected ** 2).sum(axis=2))

# Keep only unique point pairs.
pair_mask = np.triu(np.ones(dist_original.shape, dtype=bool), 1)
original_pairs = dist_original[pair_mask]
projected_pairs = dist_projected[pair_mask]

# Measure average relative distance distortion.
relative_error = np.abs(projected_pairs - original_pairs) / original_pairs
mean_error = relative_error.mean()

# Print a short summary.
print(f"Original shape: {points.shape}")
print(f"Projected shape: {projected.shape}")
print(f"Compared pairs: {original_pairs.size}")
print(f"Mean relative error: {mean_error:.3f}")

# Plot original versus projected distances.
plt.figure(figsize=(5, 5))
plt.scatter(original_pairs, projected_pairs, s=12, alpha=0.6)

# Add a perfect preservation reference line.
low = min(original_pairs.min(), projected_pairs.min())
high = max(original_pairs.max(), projected_pairs.max())

# Draw the reference line.
plt.plot([low, high], [low, high], linestyle="--")
plt.xlabel("Original pairwise distance")
plt.ylabel("Projected pairwise distance")

# Finish the teaching plot.
plt.title("Distance preserving random projection")
plt.tight_layout()
plt.show()



### **2.3. Sparse Random Projections**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_08/Lecture_B/image_02_03.jpg?v=1783839052" width="250">



>* Sparse projections cut computation using many zeros.
>* They preserve sparsity and approximate distances.

>* Preserves distances while using less computation.
>* Supports similarity tasks with acceptable accuracy.

>* Best for large, high-dimensional datasets.
>* Tune dimension and sparsity for speed-accuracy tradeoff.



In [ ]:
#@title Python Code - Sparse Random Projections

# Sparse projections can speed high dimensional transforms.
# This example preserves distances approximately after compression.
# We compare dense and sparse random matrices.

import numpy as np
import matplotlib.pyplot as plt

# Set a deterministic random seed.
rng = np.random.default_rng(7)

# Create sparse high dimensional sample data.
rows, cols, density = 80, 600, 0.04
mask = rng.random((rows, cols)) < density
X = rng.normal(size=(rows, cols)) * mask

# Validate the generated data shape.
assert X.shape == (rows, cols)

# Choose a smaller target dimension.
target_dim = 80

# Build a dense random projection matrix.
dense_matrix = rng.normal(size=(cols, target_dim)) / np.sqrt(target_dim)

# Build a sparse random projection matrix.
keep_prob = 0.1
sparse_signs = rng.choice([-1.0, 1.0], size=(cols, target_dim))
sparse_mask = rng.random((cols, target_dim)) < keep_prob
sparse_matrix = sparse_signs * sparse_mask / np.sqrt(target_dim * keep_prob)

# Project the data using both matrices.
X_dense = X @ dense_matrix
X_sparse = X @ sparse_matrix

# Pick random point pairs for distance checks.
pairs = rng.integers(0, rows, size=(25, 2))

# Measure original and projected distances.
orig_distances = np.linalg.norm(X[pairs[:, 0]] - X[pairs[:, 1]], axis=1)
dense_distances = np.linalg.norm(X_dense[pairs[:, 0]] - X_dense[pairs[:, 1]], axis=1)
sparse_distances = np.linalg.norm(X_sparse[pairs[:, 0]] - X_sparse[pairs[:, 1]], axis=1)

# Count nonzero entries in each matrix.
dense_nonzero = np.count_nonzero(dense_matrix)
sparse_nonzero = np.count_nonzero(sparse_matrix)

# Summarize approximation quality briefly.
dense_error = np.mean(np.abs(dense_distances - orig_distances) / (orig_distances + 1e-9))
sparse_error = np.mean(np.abs(sparse_distances - orig_distances) / (orig_distances + 1e-9))
print(f"Original shape: {X.shape}")
print(f"Projected shape: {X_sparse.shape}")
print(f"Dense matrix nonzeros: {dense_nonzero}")
print(f"Sparse matrix nonzeros: {sparse_nonzero}")
print(f"Dense mean relative error: {dense_error:.3f}")
print(f"Sparse mean relative error: {sparse_error:.3f}")

# Plot original versus sparse projected distances.
plt.figure(figsize=(5, 4))
plt.scatter(orig_distances, sparse_distances, color="teal", alpha=0.8)
plt.plot([0, 12], [0, 12], color="gray")
plt.xlabel("Original distance")
plt.ylabel("Sparse projected distance")
plt.title("Sparse random projection preserves distances approximately")
plt.tight_layout()
plt.show()



## **3. Approximation Tradeoffs**

### **3.1. Runtime Comparison**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_08/Lecture_B/image_03_01.jpg?v=1783839059" width="250">



>* Approximation reduces costly exact kernel computation.
>* Compare feature, training, and prediction time.

>* Approximation helps more on larger datasets.
>* Feature count affects speed, memory, accuracy.

>* Measure full pipeline under realistic conditions.
>* Balance speed, memory, and task needs.



### **3.2. Impact on Scores**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_08/Lecture_B/image_03_02.jpg?v=1783839061" width="250">



>* Scores show preserved predictive relationships after approximation.
>* Coarse mappings can blur patterns, lowering accuracy.

>* Approximation can preserve scores while cutting cost.
>* High-stakes tasks need stricter score evaluation.

>* Scores improve, then show diminishing returns.
>* Compare dimensions and runs for balanced choice.



### **3.3. Runtime Score Tradeoffs**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_08/Lecture_B/image_03_03.jpg?v=1783839063" width="250">



>* Faster approximations can reduce predictive accuracy.
>* Choose speed gains that keep performance useful.

>* Smaller dimensions run faster but lose detail.
>* Best setting depends on task needs.

>* Judge usefulness, not just one score.
>* Choose efficient methods with stable performance.



# <font color="#418FDE" size="6.5" uppercase>**Kernel Approximation**</font>


In this lecture, you learned to:
- Apply kernel approximation methods to create explicit feature maps for nonlinear models. 
- Use random projection methods to reduce dimensionality while approximately preserving distances. 
- Evaluate approximation quality by comparing runtime, transformed dimensions, and downstream scores. 

In the next Module (Module 9), we will go over 'None'